# 03 - Connectome Ablation Studies

Comparing 5 experimental conditions to evaluate the inductive bias of biological connectivity:
1. Full biological connectome
2. Random weights with identical topology
3. 50% pruned connectome (top weights)
4. 90% pruned connectome
5. Unconstrained MLP baseline


In [ ]:
import sys, os
from pathlib import Path

# Add project root and research/ to sys.path
cwd = Path.cwd().resolve()
for p in [cwd, cwd.parent, cwd.parent.parent]:
    if (p / 'research' / 'src').is_dir() and str(p) not in sys.path:
        sys.path.insert(0, str(p))
    if (p / 'src').is_dir() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

try:
    from src.config import CONFIG
    from src.connectome_utils import ConnectomeData, ConnectomeAnalysis, prune_graph_by_weight_threshold
    from src.environment import FlyNavigationEnv
    from src.agent import ConnectomeConstrainedAgent, BaselineAgent, RandomWeightAgent, PPOTrainer
    from src.metrics import MetricsLogger, analyze_neuron_activations, plot_activation_distribution, compute_path_efficiency
except ModuleNotFoundError:
    from research.src.config import CONFIG
    from research.src.connectome_utils import ConnectomeData, ConnectomeAnalysis, prune_graph_by_weight_threshold
    from research.src.environment import FlyNavigationEnv
    from research.src.agent import ConnectomeConstrainedAgent, BaselineAgent, RandomWeightAgent, PPOTrainer
    from research.src.metrics import MetricsLogger, analyze_neuron_activations, plot_activation_distribution, compute_path_efficiency
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image


## 1. Load or Run Ablation Conditions


In [ ]:
data = ConnectomeData()
neurons_df = data.fetch_all_neurons()
conn_df = data.fetch_connectivity(min_synapses=CONFIG['synapse_threshold'])
base_graph = data.build_networkx_graph(conn_df)

analysis = ConnectomeAnalysis(base_graph, neurons_df)
cx_graph = analysis.get_subgraph_by_roi('CX')
pruned_50 = prune_graph_by_weight_threshold(cx_graph, percentile=50.0)
pruned_90 = prune_graph_by_weight_threshold(cx_graph, percentile=10.0)

print(f"Full CX Graph: {cx_graph.number_of_nodes()} nodes, {cx_graph.number_of_edges()} edges")
print(f"Pruned 50% Graph: {pruned_50.number_of_edges()} edges")
print(f"Pruned 90% Graph: {pruned_90.number_of_edges()} edges")

# Load existing experiment summary if available
summary_paths = [
    Path('experiments/summary.csv'),
    Path('../experiments/summary.csv'),
    Path('../../research/experiments/summary.csv')
]

summary_file = next((p for p in summary_paths if p.exists()), None)
if summary_file:
    summary_df = pd.read_csv(summary_file, index_col=0)
    print("\nLoaded existing experiment summary:")
    display(summary_df)
else:
    print("\nRun `python scripts/run_all_experiments.py` to train all ablation conditions.")


## 2. Comparative Evaluation
Plotting comparative performance across all experimental conditions.


In [ ]:
comp_paths = [
    Path('experiments/comparison.png'),
    Path('../experiments/comparison.png'),
    Path('../../research/experiments/comparison.png')
]

comp_img = next((p for p in comp_paths if p.exists()), None)
if comp_img:
    display(Image(filename=str(comp_img)))
else:
    print("Run `python scripts/run_all_experiments.py` to generate the comparison plot.")
